# Trapped ions and phonon sidebands

**Professor Nano Labs · Quantum computing**

**Try it live in your browser:** [Trapped Ions and Phonon Sidebands lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/trapped-ion-qubit/)
**Watch first:** [How exactly does trapped ions perform quantum computing?](https://www.youtube.com/watch?v=5qpXdGSLWig)
**Watch later:** [How to trap ions for quantum computing?](https://www.youtube.com/watch?v=8p6P-QTQM2M) · [Two and three qubits quantum gates](https://www.youtube.com/watch?v=iMjpZwISIlA) · [Two qubit state: Separable vs Entangled vs Bell states](https://www.youtube.com/watch?v=4BRQsrh4oGw)

This notebook follows the video's notation. The qubit is a calcium ion, Ca⁺: $|0\rangle$ is the ground state 4S$_{1/2}$ and $|1\rangle$ the long-lived 3D$_{5/2}$ state, joined by a 729 nm transition. Inside the trap the video writes them as $|g\rangle$ and $|e\rangle$. The ions share their motion, quantized into phonons $|n_p\rangle$ spaced by $\hbar\omega_p$, so each ion's states form a ladder $|g,n_p\rangle$, $|e,n_p\rangle$, and the laser's colour picks the rungs it connects:

| laser | energy | transition | rate |
|---|---|---|---|
| carrier (green) | $\hbar\omega_0$ | $\lvert g,n_p\rangle\leftrightarrow\lvert e,n_p\rangle$ | $\Omega$ |
| blue sideband | $\hbar(\omega_0+\omega_p)$ | $\lvert g,n_p\rangle\leftrightarrow\lvert e,n_p+1\rangle$ | $\eta\sqrt{n_p+1}\,\Omega$ |
| red sideband | $\hbar(\omega_0-\omega_p)$ | $\lvert g,n_p\rangle\leftrightarrow\lvert e,n_p-1\rangle$ | $\eta\sqrt{n_p}\,\Omega$ |

The rates are the one thing the video does not need but a simulation does. $\eta$, the Lamb–Dicke parameter, says how strongly the laser's kick couples to the motion. We use $\Omega/2\pi=50$ kHz, $\eta=0.1$ and $\omega_p/2\pi=1$ MHz.

The sections follow the video's chapters and the lab's four tabs: carrier and sidebands, the phonon bus, sideband cooling, and initialization and readout. It needs only NumPy and Matplotlib. **Run all cells from the top.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
TAU = 2 * np.pi
OMEGA = TAU * 0.05      # carrier Rabi frequency in rad/us (Omega/2pi = 50 kHz)
ETA = 0.1               # Lamb-Dicke parameter
OMEGA_P = TAU * 1.0     # phonon frequency omega_p in rad/us (1 MHz)
COLOUR = {'carrier': '#3d9a4f', 'blue': '#3f74d6', 'red': '#d9495f'}
SHIFT = {'carrier': 0, 'blue': 1, 'red': -1}

def rabi(kind, n):
    # rate of |g, n> <-> |e, n + shift>
    if kind == 'carrier':
        return OMEGA
    if kind == 'blue':
        return ETA * np.sqrt(n + 1) * OMEGA
    return ETA * np.sqrt(n) * OMEGA

for kind in ('carrier', 'blue', 'red'):
    times = ', '.join(f'{np.pi / rabi(kind, n):.1f} us' if rabi(kind, n) else 'none' for n in range(3))
    print(f'{kind:8s} pi pulse from |g,0p>, |g,1p>, |g,2p>: {times}')

## 1. Carrier and sidebands

*Video, Chapter 2: carrier and phonon sideband transitions.*

In the resolved-sideband limit each laser colour drives only its own transitions, and each pair $|g,n_p\rangle\leftrightarrow|e,n_p'\rangle$ is a two-level system that flops as $\sin^2(\text{rate}\cdot t/2)$. Starting from a phonon distribution $P(n_p)$ with the ion in $|g\rangle$, the excitation probability is the sum over phonon numbers.

Look at three things in the plots: the red sideband does nothing from $|g,0_p\rangle$; sidebands are ten times slower than the carrier; and a thermal ion's sideband flopping washes out because each $n_p$ has its own rate.

In [ ]:
def thermal(nbar, N=40):
    n = np.arange(N + 1, dtype=float)
    P = float(nbar)**n / (float(nbar) + 1)**(n + 1)
    return P / P.sum()

def fock(n0, N=12):
    P = np.zeros(N + 1); P[n0] = 1
    return P

def excited(P, kind, t):
    # P(e) at time t, ion starting in |g> with phonon distribution P
    total = 0.0
    for n, p in enumerate(P):
        if p and n + SHIFT[kind] >= 0 and rabi(kind, n) > 0:
            total += p * np.sin(rabi(kind, n) * t / 2)**2
    return total

t = np.linspace(0, 400, 1601)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8), sharey=True)
for ax, (title, P) in zip(axes, [('start in |g, 0_p>', fock(0)), ('start in |g, 1_p>', fock(1)), ('thermal, mean 1 phonon', thermal(1))]):
    for kind in ('carrier', 'blue', 'red'):
        ax.plot(t, [excited(P, kind, x) for x in t], color=COLOUR[kind], lw=0.8 if kind == 'carrier' else 2,
                alpha=0.35 if kind == 'carrier' else 1, label=kind)
    ax.set_title(title); ax.set_xlabel('Pulse length t (µs)'); ax.grid(alpha=0.3)
axes[0].set_ylabel('P(e)'); axes[0].legend(loc='lower right')
plt.tight_layout(); plt.show()

pe = np.array([excited(thermal(1), 'blue', x) for x in t])
print(f'thermal mean 1 phonon, blue sideband: best P(e) = {pe.max():.3f} at {t[pe.argmax()]:.1f} us')

### Check against the full ion–phonon Hamiltonian

The model above keeps only the resonant transitions. The full Hamiltonian, in the frame rotating with the laser, keeps everything:

$$H=-\frac{\hbar\delta}{2}\sigma_z+\hbar\omega_p a^\dagger a+\frac{\hbar\Omega}{2}\left(\sigma_+e^{i\eta(a+a^\dagger)}+\text{h.c.}\right),\qquad \delta=\omega_{\rm laser}-\omega_0 .$$

The carrier agrees within 0.5%. On a sideband the off-resonant carrier shifts the line by about $\Omega^2/2\omega_p$, a *light shift*; retuning the laser by that amount, as experiments do, brings agreement within 1%.

In [ ]:
N = 12
a = np.diag(np.sqrt(np.arange(1, N + 1)), 1)
w, V = np.linalg.eigh(a + a.T)
D = V @ np.diag(np.exp(1j * ETA * w)) @ V.T             # exp(i eta (a + a^dagger))
I_N = np.eye(N + 1)
sigma_plus = np.array([[0, 0], [1, 0]])                 # |e><g|, basis order (g, e)
sigma_z = np.diag([-1, 1])

def P_excited_full(delta, n0, ts):
    H = (np.kron(-delta / 2 * sigma_z, I_N) + np.kron(np.eye(2), OMEGA_P * np.diag(np.arange(N + 1)))
         + OMEGA / 2 * (np.kron(sigma_plus, D) + np.kron(sigma_plus.T, D.conj().T)))
    e, U = np.linalg.eigh(H)
    c = U.conj().T @ np.eye(2 * (N + 1))[n0]
    return np.array([np.sum(np.abs((U @ (np.exp(-1j * e * x) * c))[N + 1:])**2) for x in ts])

light_shift = OMEGA**2 / (2 * OMEGA_P)
print(f'light shift Omega^2 / 2 omega_p = 2 pi x {light_shift / TAU * 1000:.2f} kHz')
fig, ax = plt.subplots(figsize=(8, 3.8))
for kind, delta, n0, retuned in [('carrier', 0, 0, 0), ('blue', OMEGA_P, 0, -light_shift), ('red', -OMEGA_P, 1, light_shift)]:
    tpi = np.pi / rabi(kind, n0)
    ts = np.linspace(0, tpi, 300)
    pair = np.sin(rabi(kind, n0) * ts / 2)**2
    raw = P_excited_full(delta, n0, ts)
    fixed = P_excited_full(delta + retuned, n0, ts)
    print(f'{kind:8s} from |g,{n0}p>: max difference {np.abs(raw - pair).max():.3f} at the bare sideband, '
          f'{np.abs(fixed - pair).max():.3f} after retuning by the light shift')
    ax.plot(ts / tpi, pair, color=COLOUR[kind], lw=2.5, label=f'{kind}, model')
    ax.plot(ts / tpi, fixed, '--', color='k', lw=1)
ax.set_xlabel('Pulse length / π-pulse time'); ax.set_ylabel('P(e)'); ax.set_title('Model (colour) and full Hamiltonian (black dashed)')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 2. The phonon bus

*Video, Chapter 2: two-qubit correlation and phonon-mediated coupling.*

Two ions, qubit 2 and qubit 3, share one phonon mode, which acts as a bus qubit. The state is a vector of amplitudes over $|a_2\,a_3,n_p\rangle$. A pulse rotates each pair of states its laser connects; pulse areas refer to the computational transitions ($|g,0_p\rangle\leftrightarrow|e,1_p\rangle$ for blue, $|g,1_p\rangle\leftrightarrow|e,0_p\rangle$ for red), so other phonon numbers turn faster by $\sqrt{n_p+1}$ or $\sqrt{n_p}$.

The video's rule: with the bus in $|0_p\rangle$ the red laser lowers a qubit and the blue raises it; with the bus in $|1_p\rangle$ they swap roles.

In [ ]:
NB = 12                                                   # bus kept up to 12 phonons
def idx(a2, a3, n):
    return (a2 * 2 + a3) * (NB + 1) + n

def bus_state(a2, a3, n):
    psi = np.zeros(4 * (NB + 1), complex); psi[idx(a2, a3, n)] = 1
    return psi

def pulse(psi, ion, kind, area):
    out = psi.copy()
    for other in (0, 1):
        for n in range(NB + 1):
            m = n + SHIFT[kind]
            if m < 0 or m > NB:
                continue
            factor = 1 if kind == 'carrier' else np.sqrt(n + 1) if kind == 'blue' else np.sqrt(n)
            if factor == 0:
                continue
            c, s = np.cos(area * factor / 2), np.sin(area * factor / 2)
            lo = idx(0, other, n) if ion == 2 else idx(other, 0, n)
            hi = idx(1, other, m) if ion == 2 else idx(other, 1, m)
            G, E = psi[lo], psi[hi]
            if kind == 'carrier':                         # cos - i sin sigma_x
                out[lo], out[hi] = c * G - 1j * s * E, -1j * s * G + c * E
            else:                                         # sideband coupling i*eta*(...): a real rotation
                out[lo], out[hi] = c * G - s * E, s * G + c * E
    return out

def run(start, pulses):
    psi = bus_state(*start)
    for p in pulses:
        psi = pulse(psi, *p)
    return psi

def show(psi):
    terms = []
    for a2 in (0, 1):
        for a3 in (0, 1):
            for n in range(NB + 1):
                z = psi[idx(a2, a3, n)]
                if abs(z) > 1e-6:
                    terms.append(f'({z.real:+.3f}{z.imag:+.3f}i)|{"ge"[a2]}2 {"ge"[a3]}3, {n}p>')
    return '  '.join(terms)

def concurrence_if_bus_definite(psi):
    phon = [sum(abs(psi[idx(a2, a3, n)])**2 for a2 in (0, 1) for a3 in (0, 1)) for n in range(NB + 1)]
    n = int(np.argmax(phon))
    if phon[n] < 1 - 1e-9:
        return None
    c00, c01, c10, c11 = (psi[idx(a2, a3, n)] for a2, a3 in ((0, 0), (0, 1), (1, 0), (1, 1)))
    return 2 * abs(c00 * c11 - c01 * c10)

pi = np.pi
print('bus |0p>, red pi on qubit 3:  |g2 e3,0p> ->', show(run((0, 1, 0), [(3, 'red', pi)])))
print('bus |0p>, blue pi on qubit 3: |g2 g3,0p> ->', show(run((0, 0, 0), [(3, 'blue', pi)])))
print('bus |1p>, red pi on qubit 3:  |g2 g3,1p> ->', show(run((0, 0, 1), [(3, 'red', pi)])))
print('bus |1p>, blue pi on qubit 3: |g2 e3,1p> ->', show(run((0, 1, 1), [(3, 'blue', pi)])))

transfer = [(2, 'carrier', pi / 2), (2, 'red', pi), (3, 'red', pi)]
entangle = [(2, 'carrier', pi), (2, 'red', pi / 2), (3, 'red', pi)]
print('\nmove qubit 2 onto qubit 3:', show(run((0, 0, 0), transfer)))
psi_e = run((0, 0, 0), entangle)
print('entangle qubits 2 and 3:  ', show(psi_e), '  concurrence', concurrence_if_bus_definite(psi_e))
psi_w = run((0, 0, 1), entangle)
print('same recipe, warm bus:    ', show(psi_w), '  concurrence', concurrence_if_bus_definite(psi_w))

## 3. Sideband cooling

*Video, Chapter 3: sideband cooling of qubits.*

Each cycle, a red-sideband pulse moves $|g,n_p\rangle$ to $|e,n_p-1\rangle$ with probability $\sin^2(\theta\sqrt{n_p}/2)$; the ion then emits a photon and returns to $|g\rangle$, keeping $n_p-1$ except for a small recoil that adds or removes one more phonon. Repeating pumps the phonons out.

A pulse that is a π pulse for $1_p$ is a 2π pulse for $4_p$ (because $\sqrt4=2$), so population at $4_p$ never moves: a *dark state*. Varying the pulse length fixes it.

In [ ]:
def cool_cycle(P, tuned, recoil):
    area = np.pi / np.sqrt(tuned)
    nxt = np.zeros_like(P)
    N = len(P) - 1
    for n, p in enumerate(P):
        moved = p * np.sin(area * np.sqrt(n) / 2)**2 if n > 0 else 0.0
        nxt[n] += p - moved
        if moved:
            m = n - 1
            down = recoil if m > 0 else 0.0
            nxt[m] += moved * (1 - down - recoil)
            nxt[min(N, m + 1)] += moved * recoil
            if down:
                nxt[m - 1] += moved * down
    return nxt

PLANS = {'fixed': [1], 'varied': [1, 2, 4]}
def cool(P0, cycles, plan, recoil=0.01):
    hist = [P0]
    for k in range(cycles):
        hist.append(cool_cycle(hist[-1], PLANS[plan][k % len(PLANS[plan])], recoil))
    return hist

P0 = thermal(5, 80)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
n = np.arange(21)
a1.bar(n, P0[:21], width=0.8, fill=False, edgecolor='0.4', label='after Doppler cooling (mean 5)')
for plan, col in (('fixed', '#d9495f'), ('varied', '#3d9a4f')):
    hist = cool(P0, 80, plan)
    a1.bar(n + (0.2 if plan == 'varied' else -0.2), hist[40][:21], width=0.4, color=col, alpha=0.8, label=f'{plan} pulses, 40 cycles')
    means = [np.dot(np.arange(len(P)), P) for P in hist]
    a2.plot(means, color=col, lw=2, label=f'{plan}: mean phonons')
    a2.plot([P[0] for P in hist], color=col, lw=2, ls='--', label=f'{plan}: P(0_p)')
    print(f'{plan:6s}: P(0p) after 40 cycles {hist[40][0]:.3f}, after 80 {hist[80][0]:.4f}; stuck in 4p after 40: {hist[40][4]:.3f}')
a1.set_xlabel('Phonon number n_p'); a1.set_ylabel('Probability'); a1.legend(fontsize=8)
a2.set_xlabel('Cooling cycle'); a2.legend(fontsize=8); a2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# the red/blue sideband ratio is a thermometer: <n> / (<n> + 1)
for k in (0, 40):
    P = cool(P0, 40, 'varied')[k]
    nbar = np.dot(np.arange(len(P)), P)
    print(f'after {k:2d} cycles: mean phonons {nbar:.3f}, red/blue sideband ratio {nbar / (nbar + 1):.3f}')

## 4. Initialization and readout

*Video, Chapter 4: qubit encoding, initialization and readout.*

**Optical pumping** empties $|1\rangle$ through the short-lived 4P level, which decays to $|0\rangle$; each cycle returns about 94% of the $|1\rangle$ population (roughly the branching of the 4P$_{3/2}$ decay in Ca⁺). A 729 nm carrier pulse of area $\theta$ then leaves $\cos(\theta/2)|0\rangle-i\sin(\theta/2)|1\rangle$.

**Fluorescence readout:** a laser on the 4S–4P transition makes the ion glow only in $|0\rangle$. Each shot gives a Poisson number of detector clicks, with mean $\mu_b+\mu_d$ for a bright ion and $\mu_d$ (background) for a dark one. A threshold turns clicks into bright or dark; repeating gives the probability.

In [ ]:
rng = np.random.default_rng(3)
for k in range(5):
    print(f'{k} optical-pumping cycles: P(1) left = {0.5 * 0.06**k:.2e}')

def poisson_cdf(mu, kmax):
    if kmax < 0:
        return 0.0
    k = np.arange(kmax + 1)
    logp = -mu + k * np.log(mu) - np.cumsum(np.log(np.maximum(k, 1)))
    return float(np.exp(logp).sum())

def readout_errors(mu_b, mu_d, threshold):
    return {'dark read as bright': 1 - poisson_cdf(mu_d, threshold - 1),
            'bright read as dark': poisson_cdf(mu_b + mu_d, threshold - 1)}

def shots(P0, mu_b, mu_d, n):
    bright = rng.random(n) < P0
    return np.where(bright, rng.poisson(mu_b + mu_d, n), rng.poisson(mu_d, n))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, (mu_b, thr) in zip(axes, [(3, 2), (8, 3)]):
    c = shots(0.5, mu_b, 0.3, 5000)
    ax.hist(c, bins=np.arange(-0.5, 21.5), color='#6b4bd1', alpha=0.75)
    ax.axvline(thr - 0.5, color='#c47a12', ls='--', label='threshold')
    err = readout_errors(mu_b, 0.3, thr)
    ax.set_title(f'{mu_b} photons when bright: bright fraction {np.mean(c >= thr):.3f}')
    ax.set_xlabel('Detector clicks in one shot'); ax.legend()
    print(f'mu_b = {mu_b}, threshold {thr}:', {k: round(v, 4) for k, v in err.items()})
plt.tight_layout(); plt.show()

## 5. Checks

The numbers quoted in the lab's "Predict first" answers, recomputed here.

In [ ]:
# Q1: no red sideband from |g, 0p>
assert rabi('red', 0) == 0
# Q2: pi pulses 10, 100 and 70.7 us
assert np.isclose(np.pi / rabi('carrier', 0), 10) and np.isclose(np.pi / rabi('blue', 0), 100)
assert np.isclose(np.pi / rabi('blue', 1), 100 / np.sqrt(2))
# Q3: thermal mean 1, blue sideband peaks near 0.82 at 72 us
assert abs(pe.max() - 0.817) < 0.002 and abs(t[pe.argmax()] - 72) < 1
# Q4: red lowers with the bus in 0p and raises with it in 1p
assert abs(abs(run((0, 1, 0), [(3, 'red', pi)])[idx(0, 0, 1)]) - 1) < 1e-12
assert abs(abs(run((0, 0, 1), [(3, 'red', pi)])[idx(0, 1, 0)]) - 1) < 1e-12
# Q5: the entangling recipe gives a Bell state with the bus back in 0p
assert np.isclose(concurrence_if_bus_definite(psi_e), 1)
# Q6: with a warm bus the bus stays entangled
assert concurrence_if_bus_definite(psi_w) is None
# Q7: fixed pulses stall near half in 0p with about 40% stuck in 4p; varied pulses reach ~97% in 40 cycles
h_fixed, h_var = cool(P0, 80, 'fixed'), cool(P0, 80, 'varied')
assert abs(h_fixed[40][0] - 0.52) < 0.01 and abs(h_fixed[40][4] - 0.41) < 0.01
assert abs(h_var[40][0] - 0.965) < 0.005 and h_var[80][0] > 0.99
# Q8: dim detector errors, and about 44% called bright for an equal superposition
e = readout_errors(3, 0.3, 2)
assert abs(e['bright read as dark'] - 0.159) < 0.001 and abs(e['dark read as bright'] - 0.037) < 0.001
assert abs(0.5 * (1 - e['bright read as dark']) + 0.5 * e['dark read as bright'] - 0.44) < 0.003
print('All checks passed.')

## Things to try

Each idea matches a "Predict first" question in the [browser lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/trapped-ion-qubit/).

1. **Which colour moves the phonons?** Plot the phonon distribution before and after a blue π pulse from $|g,0_p\rangle$, and after a carrier π pulse.
2. **Warm sidebands.** Plot blue-sideband flopping for thermal ions with mean 0.1, 0.5 and 2 phonons. How does the wash-out change?
3. **A different recipe.** Can you entangle qubits 2 and 3 using blue pulses instead of red ones? (Hint: what does blue do with the bus in $|1_p\rangle$?)
4. **More dark states.** For pulses tuned to $1_p$, which other phonon numbers are stuck? Check $n_p=9$ and $n_p=16$.
5. **Readout threshold.** For $\mu_b=5$, find the threshold that minimizes the average of the two readout errors.